# Part 2: Finding similar users with Shingling, MinHash and LSH

The dataset has about 206,000 users. Comparing every user with every other user means n(n-1)/2, or roughly 2.1 x 10^10 pairs. Computing an exact Jaccard similarity for every one of those pairs is too much work, so instead we approximate: MinHash compresses each user into a short signature, and Locality-Sensitive Hashing (LSH) only proposes the pairs that are likely to be similar. We then compute the exact similarity on those candidate pairs only.

The output of this notebook is similar_users.parquet: the top-K most similar users for each user, used for the recommendation and evaluation step.

In [ ]:
# Load the prior-only user product sets and product lookup table from preprocessing.

# Requires pip install pyarrow numpy pandas scipy matplotlib

import os
import time
import tracemalloc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

save_path = "processed/"
results_path = "results/"     # Experiment result tables are saved here as .csv files.

print ("~~~~ Loading... ~~~~")

user_products = pd.read_parquet(os.path.join(save_path, "user_prior_products.parquet"), engine = "pyarrow")
products_lookup = pd.read_parquet(os.path.join(save_path, "products_lookup.parquet"), engine = "pyarrow")

print ("\t user_prior_products has shape:\t", user_products.shape)
print ("\t products_lookup has shape:\t", products_lookup.shape)
print ("~~~~ Load complete. ~~~~")

n_total_users = len(user_products)
print ("\nUsers:", n_total_users, "\tPairs to compare exactly:", f"{n_total_users * (n_total_users - 1) // 2:.2e}")

display (user_products.head())

## Shingles

Each user becomes the set of product IDs from their prior orders. This is the "collaborative filtering as a similar-sets problem" case from section 3.1.3 of the textbook: two customers are similar if the sets of items they bought overlap a lot.

k-character shingles do not apply here. Shingling by characters is meant for documents, where the order of characters and words carries meaning. A user's purchases are a set of atomic items with no order, and the characters of a product's name say nothing about who buys it ("Organic Banana" and "Organic Bandages" share many shingles but are unrelated). Each product ID is therefore already the natural "shingle".

As an optional, coarser version, each user can instead be represented by the set of aisles they bought from (set USE_AISLE_SETS = True). Aisle sets are much smaller and overlap more, so similarities will be higher.

Users with very few distinct products have very noisy signatures: with only 2 or 3 items, a single shared product swings the Jaccard similarity a lot. We drop these users from the similarity search.

In [ ]:
# Build one set of item IDs per user, and drop users with too few distinct items.

MIN_PRODUCTS = 5          # Users with fewer distinct items than this have noisy signatures and are dropped.
USE_AISLE_SETS = False    # Optional coarser version: use the set of aisles each user bought from instead.

if USE_AISLE_SETS:
  # Map each product ID to an integer aisle code, then rebuild each user's set from those codes.
  aisle_codes = pd.Series(pd.factorize(products_lookup["aisle"])[0].astype(np.int32), index = products_lookup["product_id"])
  exploded = user_products[["user_id", "product_id"]].explode("product_id")
  exploded["item_id"] = exploded["product_id"].astype(np.int32).map(aisle_codes)
  user_items = exploded[["user_id", "item_id"]].drop_duplicates().groupby("user_id", as_index = False)["item_id"].agg(list)
  del exploded
else:
  user_items = user_products[["user_id", "product_id"]].rename(columns = {"product_id": "item_id"})

# Convert each list to a set (np.unique removes repeats and sorts). Jaccard works on sets, so a product
# bought 10 times must only count once.
item_sets = [np.unique(np.asarray(items, dtype = np.int32)) for items in user_items["item_id"]]
all_set_sizes = np.array([len(items) for items in item_sets], dtype = np.int32)

keep = all_set_sizes >= MIN_PRODUCTS
user_ids = user_items["user_id"].to_numpy()[keep].astype(np.int32)
item_sets = [items for items, k in zip(item_sets, keep) if k]

# Store all sets in one flat array plus a start offset per user, so numpy can process many users at once.
set_sizes = all_set_sizes[keep]
flat_items = np.concatenate(item_sets)
offsets = np.r_[0, np.cumsum(set_sizes)[:-1]].astype(np.int64)
n_users = len(user_ids)
del item_sets

# Sparse user x item matrix, used to compute exact Jaccard similarities quickly.
X = sparse.csr_matrix((np.ones(len(flat_items), dtype = np.int32), flat_items, np.r_[offsets, len(flat_items)]),
                      shape = (n_users, int(flat_items.max()) + 1))

print ("~~~~ User sets ~~~~")
print ("Users dropped (fewer than", MIN_PRODUCTS, "items):\t", int((~keep).sum()))
print ("Users kept:\t\t\t\t", n_users)
print ("Distinct items:\t\t\t\t", len(np.unique(flat_items)))
print ("Items per user:\n", pd.Series(set_sizes).describe())

## MinHash

MinHash replaces each user's set with a short signature. For each of the hash functions h(x) = (a·x + b) mod p, the signature stores the smallest hash value among the user's items. The probability that two users have the same value in one row of the signature equals their Jaccard similarity, so the fraction of matching rows is an estimate of it.

We start with 120 hash functions, with p = 2^31 - 1 (a large prime). To keep this fast, every possible item ID is hashed once into a lookup table, and users are processed in chunks with numpy. np.minimum.reduceat then takes the minimum over each user's items in one vectorized step. The full signature matrix is about 206K x 120 int32 values, roughly 100 MB.

In [ ]:
# MinHash: build the signature matrix with hash functions h(x) = (a*x + b) mod p.

N_HASHES = 120
PRIME = 2_147_483_647     # 2^31 - 1, a large prime. Hash values are below it, so they fit in int32.
CHUNK_USERS = 2000        # Users processed per chunk, to keep memory use small.
SEED = 612

def make_hash_params(n_hashes, seed = SEED):
  # Random coefficients a (non-zero) and b for each hash function.
  rng = np.random.default_rng(seed)
  a = rng.integers(1, PRIME, size = n_hashes, dtype = np.int64)
  b = rng.integers(0, PRIME, size = n_hashes, dtype = np.int64)
  return a, b

def minhash_signatures(flat_items, offsets, set_sizes, n_hashes, chunk_users = CHUNK_USERS, seed = SEED):
  # Returns an (n_users x n_hashes) int32 signature matrix.
  a, b = make_hash_params(n_hashes, seed)

  # Hash every possible item ID once. a < 2^31 and item IDs are small, so a*x fits in int64.
  all_items = np.arange(int(flat_items.max()) + 1, dtype = np.int64)
  hash_table = ((all_items[:, None] * a[None, :] + b[None, :]) % PRIME).astype(np.int32)

  signatures = np.empty((len(set_sizes), n_hashes), dtype = np.int32)
  for start in range(0, len(set_sizes), chunk_users):
    stop = min(start + chunk_users, len(set_sizes))
    item_start = offsets[start]
    item_stop = offsets[stop - 1] + set_sizes[stop - 1]

    # Hash values of every item in this chunk, then the minimum over each user's items.
    hashed = hash_table[flat_items[item_start:item_stop]]
    signatures[start:stop] = np.minimum.reduceat(hashed, offsets[start:stop] - item_start, axis = 0)

  return signatures

start_time = time.perf_counter()
signatures = minhash_signatures(flat_items, offsets, set_sizes, N_HASHES)

print ("~~~~ MinHash complete ~~~~")
print ("Signature matrix has shape:\t", signatures.shape)
print ("Size in RAM:\t\t\t", round(signatures.nbytes / 1e6, 1), "MB")
print ("Time:\t\t\t\t", round(time.perf_counter() - start_time, 2), "seconds")

## LSH banding

The signature matrix is split into b bands of r rows each (b·r = 120). Two users become a candidate pair if all r rows match in at least one band. For a pair with Jaccard similarity s, the probability of becoming a candidate is 1 - (1 - s^r)^b, which forms an S-curve. The threshold where the curve rises steeply is about (1/b)^(1/r).

Grocery customers only overlap loosely, so we expect true Jaccard values of roughly 0.1 to 0.4. With 40 bands of 3 rows, the threshold is (1/40)^(1/3) ~= 0.29, which is a sensible starting point. The plot below compares the S-curves of several (b, r) choices that all use 120 hash functions.

In [ ]:
# Plot the LSH S-curve, 1 - (1 - s^r)^b, for several (b, r) choices.

BAND_SETTINGS = [(60, 2), (40, 3), (30, 4), (24, 5), (20, 6)]    # All use b * r = 120 hash functions.

similarity = np.linspace(0, 1, 201)

plt.figure(figsize = (8, 5))
for bands, rows in BAND_SETTINGS:
  threshold = (1 / bands) ** (1 / rows)
  plt.plot(similarity, 1 - (1 - similarity ** rows) ** bands, label = f"b = {bands}, r = {rows}  (threshold ~= {threshold:.2f})")
plt.axvspan(0.1, 0.4, color = "grey", alpha = 0.15, label = "Expected grocery Jaccard range")
plt.xlabel("Jaccard similarity of a pair")
plt.ylabel("Probability of becoming a candidate pair")
plt.title("LSH S-curves")
plt.legend()
plt.grid(alpha = 0.3)
plt.show()

Very common products, like bananas, are the minimum hash value for many users at once. This puts thousands of users into the same bucket, which creates millions of candidate pairs that are not actually similar. To prevent this, we skip any bucket with more than MAX_BUCKET_SIZE users.

Each user is identified by its row position in the signature matrix, and each candidate pair is stored as a single int64 (u * n_users + v, with u < v) so duplicates found in several bands can be removed with np.unique.

In [ ]:
# LSH banding: hash each band of the signatures into buckets, and collect candidate pairs from each bucket.

BANDS = 40
ROWS = 3
MAX_BUCKET_SIZE = 300     # Skip buckets larger than this (caused by very common products like bananas).

def band_candidate_pairs(band, max_bucket_size):
  # Candidate pairs (encoded as u * n + v, u < v) for one band of shape (n_users x rows).
  n = len(band)

  # Users with identical rows in this band share a bucket. np.unique gives each bucket an integer label.
  _, labels = np.unique(band, axis = 0, return_inverse = True)
  labels = labels.ravel()
  order = np.argsort(labels, kind = "stable")
  sorted_labels = labels[order]
  bucket_starts = np.flatnonzero(np.r_[True, sorted_labels[1:] != sorted_labels[:-1]])
  bucket_sizes = np.diff(np.r_[bucket_starts, n])

  pairs = []
  skipped = int((bucket_sizes > max_bucket_size).sum())

  # Buckets of the same size are handled together, so all pairs inside them are made in one vectorized step.
  for size in np.unique(bucket_sizes):
    if size < 2 or size > max_bucket_size:
      continue
    starts = bucket_starts[bucket_sizes == size]
    members = order[starts[:, None] + np.arange(size)]        # (number of buckets x size) user positions
    i, j = np.triu_indices(size, k = 1)
    u = members[:, i].ravel().astype(np.int64)
    v = members[:, j].ravel().astype(np.int64)
    pairs.append(np.minimum(u, v) * n + np.maximum(u, v))

  pairs = np.unique(np.concatenate(pairs)) if pairs else np.empty(0, dtype = np.int64)
  return pairs, skipped

def lsh_candidate_pairs(signatures, bands, rows, max_bucket_size = MAX_BUCKET_SIZE):
  # Returns arrays (u, v) of candidate pairs (row positions, u < v), and the number of skipped buckets.
  assert bands * rows <= signatures.shape[1], "Need at least bands * rows hash functions."
  n = len(signatures)
  all_pairs = []
  total_skipped = 0

  for band_index in range(bands):
    band = signatures[:, band_index * rows:(band_index + 1) * rows]
    pairs, skipped = band_candidate_pairs(band, max_bucket_size)
    all_pairs.append(pairs)
    total_skipped += skipped

    # Merge every 10 bands, so duplicate pairs from different bands do not pile up in RAM.
    if len(all_pairs) >= 10:
      all_pairs = [np.unique(np.concatenate(all_pairs))]

  pairs = np.unique(np.concatenate(all_pairs))
  return pairs // n, pairs % n, total_skipped

start_time = time.perf_counter()
cand_u, cand_v, skipped_buckets = lsh_candidate_pairs(signatures, BANDS, ROWS)

print ("~~~~ LSH complete ~~~~")
print ("Bands:", BANDS, "\tRows per band:", ROWS, "\tThreshold ~=", round((1 / BANDS) ** (1 / ROWS), 3))
print ("Candidate pairs:\t\t", len(cand_u))
print ("Share of all pairs:\t\t", f"{len(cand_u) / (n_users * (n_users - 1) / 2):.2e}")
print ("Oversized buckets skipped:\t", skipped_buckets)
print ("Time:\t\t\t\t", round(time.perf_counter() - start_time, 2), "seconds")

## Checking candidates

LSH candidates include some false positives, so (as in section 3.4.3 of the textbook) we compute the exact Jaccard similarity on the candidate pairs only. This is far cheaper than computing it on all pairs.

The intersection size of each pair is computed with the sparse user x item matrix, and the union size is |A| + |B| - |A ∩ B|. We then keep the TOP_K most similar neighbours for each user and save them as similar_users.parquet for the recommendation and evaluation step.

In [ ]:
# Compute exact Jaccard similarity on the candidate pairs only, and keep the top-K neighbours per user.

TOP_K = 20
MIN_JACCARD = 0.05        # Ignore candidate pairs that turn out to be barely similar.

def exact_jaccard(X, set_sizes, u, v, chunk_size = 100_000):
  # Pairs are processed in chunks, so peak memory stays roughly constant however many candidates there are.
  # Exact Jaccard similarity |A ∩ B| / |A ∪ B| for each pair of row positions (u[i], v[i]).
  result = np.empty(len(u), dtype = np.float32)
  for start in range(0, len(u), chunk_size):
    stop = start + chunk_size
    intersection = np.asarray(X[u[start:stop]].multiply(X[v[start:stop]]).sum(axis = 1)).ravel()
    union = set_sizes[u[start:stop]] + set_sizes[v[start:stop]] - intersection
    result[start:stop] = intersection / union
  return result

start_time = time.perf_counter()
cand_jaccard = exact_jaccard(X, set_sizes, cand_u, cand_v)

# Each pair is a neighbour for both users, so list it in both directions before ranking.
similar = cand_jaccard >= MIN_JACCARD
similar_users = pd.DataFrame({
    "user_id": user_ids[np.r_[cand_u[similar], cand_v[similar]]],
    "neighbor_user_id": user_ids[np.r_[cand_v[similar], cand_u[similar]]],
    "jaccard": np.r_[cand_jaccard[similar], cand_jaccard[similar]]})

similar_users = similar_users.sort_values(["user_id", "jaccard"], ascending = [True, False])
similar_users["rank"] = (similar_users.groupby("user_id").cumcount() + 1).astype(np.int32)
similar_users = similar_users[similar_users["rank"] <= TOP_K].reset_index(drop = True)

print ("~~~~ Candidate check complete ~~~~")
print ("Candidate pairs with Jaccard >=", MIN_JACCARD, ":\t", int(similar.sum()), "of", len(cand_u))
print ("Users with at least one neighbour:\t", similar_users["user_id"].nunique(), "of", n_users)
print ("Neighbour Jaccard values:\n", similar_users["jaccard"].describe())
print ("Time:\t\t\t\t\t", round(time.perf_counter() - start_time, 2), "seconds")

display (similar_users.head(10))

In [ ]:
# Save the top-K similar users for the recommendation and evaluation step.

similar_users.to_parquet(os.path.join(save_path, "similar_users.parquet"))
print ("Saved", len(similar_users), "rows to", os.path.join(save_path, "similar_users.parquet"))

## Experiment 1: MinHash accuracy vs. number of hash functions

The MinHash estimate of a pair's Jaccard similarity is the fraction of signature rows that match. With k hash functions its standard error is about sqrt(J(1-J)/k), so more hash functions should give more accurate estimates at the cost of more time and memory.

We compare the estimate with the exact Jaccard similarity for 32, 64, 128 and 256 hash functions. Random pairs of users are almost never similar, so the sample mixes random pairs with LSH candidate pairs to cover a wider range of similarities.

In [ ]:
# Experiment 1: how far MinHash estimates are from exact Jaccard, for 32, 64, 128 and 256 hash functions.

HASH_COUNTS = [32, 64, 128, 256]
N_SAMPLE_PAIRS = 5000
rng_exp1 = np.random.default_rng(SEED)    # Each experiment has its own generator, so results do not depend on cell order.

def subset_sets(rows):
  # Flat items, offsets and sizes for a subset of users (row positions), in the same format as the full data.
  sizes = set_sizes[rows]
  items = np.concatenate([flat_items[offsets[r]:offsets[r] + set_sizes[r]] for r in rows])
  return items, np.r_[0, np.cumsum(sizes)[:-1]].astype(np.int64), sizes

# Half random pairs, half LSH candidate pairs.
random_u = rng_exp1.integers(0, n_users, N_SAMPLE_PAIRS)
random_v = rng_exp1.integers(0, n_users, N_SAMPLE_PAIRS)
not_same = random_u != random_v
picked = rng_exp1.choice(len(cand_u), size = min(N_SAMPLE_PAIRS, len(cand_u)), replace = False)
pair_u = np.r_[random_u[not_same], cand_u[picked]]
pair_v = np.r_[random_v[not_same], cand_v[picked]]
true_jaccard = exact_jaccard(X, set_sizes, pair_u, pair_v)

# Only compute signatures for the users that appear in the sampled pairs.
sample_rows, inverse = np.unique(np.r_[pair_u, pair_v], return_inverse = True)
sample_u, sample_v = inverse[:len(pair_u)], inverse[len(pair_u):]
sample_items, sample_offsets, sample_sizes = subset_sets(sample_rows)

results = []
fig, axes = plt.subplots(1, len(HASH_COUNTS), figsize = (4 * len(HASH_COUNTS), 4), sharey = True)
for ax, n_hashes in zip(axes, HASH_COUNTS):
  sample_signatures = minhash_signatures(sample_items, sample_offsets, sample_sizes, n_hashes)
  estimate = (sample_signatures[sample_u] == sample_signatures[sample_v]).mean(axis = 1)
  error = estimate - true_jaccard
  results.append({"hash_functions": n_hashes,
                  "mean_abs_error": np.abs(error).mean(),
                  "rmse": np.sqrt((error ** 2).mean()),
                  "max_abs_error": np.abs(error).max()})

  ax.scatter(true_jaccard, estimate, s = 3, alpha = 0.3)
  ax.plot([0, 1], [0, 1], color = "red", linewidth = 1)
  ax.set_title(f"{n_hashes} hash functions")
  ax.set_xlabel("Exact Jaccard")
axes[0].set_ylabel("MinHash estimate")
plt.tight_layout()
plt.show()

minhash_results = pd.DataFrame(results)

print ("~~~~ MinHash estimate error ~~~~")
print ("Pairs sampled:", len(pair_u))
display (minhash_results)

os.makedirs(results_path, exist_ok = True)
minhash_results.to_csv(os.path.join(results_path, "minhash_error.csv"), index = False)

## Experiment 2: LSH recall and false positives against brute force

On a sample of users, we compute the exact Jaccard similarity of every pair by brute force, and treat pairs at or above a similarity threshold as the truly similar ones. Since grocery similarities are low, we report several thresholds (0.1, 0.2 and 0.3). For each (b, r) setting and threshold we then measure:

- Recall: the share of truly similar pairs that LSH found as candidates.
- Precision: the share of candidate pairs that are truly similar.
- False-positive share: the share of candidate pairs that are not truly similar (1 - precision). This is not the statistical false positive rate FP / (FP + TN), which would be tiny here because almost all pairs are true negatives.
- Candidate share: the share of all pairs LSH asked us to check exactly (lower means less work).

In [ ]:
# Experiment 2: LSH recall, precision and false-positive share vs. brute force, on a sample of users.

N_SAMPLE_USERS = 5000
SIM_THRESHOLDS = [0.1, 0.2, 0.3]
rng_exp2 = np.random.default_rng(SEED + 1)

sample_rows = np.sort(rng_exp2.choice(n_users, size = min(N_SAMPLE_USERS, n_users), replace = False))
X_sample = X[sample_rows]
sample_set_sizes = set_sizes[sample_rows]
n_sample = len(sample_rows)

# Brute force: intersection sizes of every pair at once with a sparse matrix product.
intersections = (X_sample @ X_sample.T).toarray()
i, j = np.triu_indices(n_sample, k = 1)
brute_jaccard = intersections[i, j] / (sample_set_sizes[i] + sample_set_sizes[j] - intersections[i, j])
del intersections

# Truly similar pairs for each threshold, encoded as i * n_sample + j like the LSH candidates.
true_pairs = {t: set((i[brute_jaccard >= t] * n_sample + j[brute_jaccard >= t]).tolist()) for t in SIM_THRESHOLDS}

print ("Sample users:", n_sample, "\tAll pairs:", len(brute_jaccard))
for t in SIM_THRESHOLDS:
  print ("\tTruly similar pairs (Jaccard >=", t, "):", len(true_pairs[t]))

sample_signatures = signatures[sample_rows]
results = []
for bands, rows in BAND_SETTINGS:
  u, v, _ = lsh_candidate_pairs(sample_signatures, bands, rows)
  candidates = set((u * n_sample + v).tolist())
  for t in SIM_THRESHOLDS:
    found = len(candidates & true_pairs[t])
    results.append({"bands": bands, "rows": rows,
                    "lsh_threshold": round((1 / bands) ** (1 / rows), 3),
                    "sim_threshold": t,
                    "candidates": len(candidates),
                    "candidate_share": len(candidates) / len(brute_jaccard),
                    "recall": found / len(true_pairs[t]) if true_pairs[t] else np.nan,
                    "precision": found / len(candidates) if candidates else np.nan,
                    "false_positive_share": (len(candidates) - found) / len(candidates) if candidates else np.nan})

lsh_results = pd.DataFrame(results)
print ("~~~~ LSH vs. brute force ~~~~")
display (lsh_results.pivot(index = ["bands", "rows", "lsh_threshold", "candidates"], columns = "sim_threshold",
                           values = ["recall", "precision", "false_positive_share"]).round(3))

os.makedirs(results_path, exist_ok = True)
lsh_results.to_csv(os.path.join(results_path, "lsh_vs_brute_force.csv"), index = False)

## Experiment 2b: Are the pairs dropped by the bucket cap actually similar?

lsh_candidate_pairs skips any bucket with more than MAX_BUCKET_SIZE users. Experiment 2 uses a 5,000-user sample, where no bucket ever gets that large, so its recall numbers do not show the effect of the cap. On the full data the situation is different: with the default settings, about 35 buckets exceed the cap, and together they hold about 27% of all candidate pair slots (a "slot" is one pair inside one bucket of one band, so the same pair can be counted in several bands).

If those dropped pairs were truly similar users, the cap would be hurting recall. To check this, we go through every band of the full signature matrix, find the oversized buckets, and count how many pair slots are dropped and kept. We then sample up to PAIRS_PER_BUCKET random pairs from each oversized bucket and compute their exact Jaccard similarity.

In [ ]:
# Experiment 2b: check whether the candidate pairs dropped by the bucket cap are actually similar.

PAIRS_PER_BUCKET = 300
rng_cap = np.random.default_rng(SEED + 3)
lsh_threshold = (1 / BANDS) ** (1 / ROWS)

oversized_buckets = 0
dropped_slots = np.int64(0)
kept_slots = np.int64(0)
cap_u = []
cap_v = []

for band_index in range(BANDS):
  band = signatures[:, band_index * ROWS:(band_index + 1) * ROWS]

  # Same bucket labels as in band_candidate_pairs: users with identical rows in this band share a bucket.
  _, labels = np.unique(band, axis = 0, return_inverse = True)
  labels = labels.ravel()
  bucket_sizes = np.bincount(labels).astype(np.int64)
  big_buckets = np.flatnonzero(bucket_sizes > MAX_BUCKET_SIZE)
  kept_sizes = bucket_sizes[(bucket_sizes >= 2) & (bucket_sizes <= MAX_BUCKET_SIZE)]

  # A bucket of size s holds s * (s - 1) / 2 pair slots.
  oversized_buckets += len(big_buckets)
  dropped_slots += (bucket_sizes[big_buckets] * (bucket_sizes[big_buckets] - 1) // 2).sum()
  kept_slots += (kept_sizes * (kept_sizes - 1) // 2).sum()

  # Sample random pairs of members from each oversized bucket, removing pairs of a user with itself.
  for label in big_buckets:
    members = np.flatnonzero(labels == label)
    u = rng_cap.choice(members, size = PAIRS_PER_BUCKET)
    v = rng_cap.choice(members, size = PAIRS_PER_BUCKET)
    different = u != v
    cap_u.append(u[different])
    cap_v.append(v[different])

total_slots = dropped_slots + kept_slots
dropped_share = dropped_slots / total_slots if total_slots else 0.0

print ("~~~~ Bucket cap check ~~~~")
print ("Bands:", BANDS, "\tRows per band:", ROWS, "\tMax bucket size:", MAX_BUCKET_SIZE)
print ("Oversized buckets:\t", oversized_buckets)
print ("Pair slots dropped:\t", int(dropped_slots))
print ("Pair slots kept:\t", int(kept_slots))
print ("Dropped share:\t\t", round(float(dropped_share), 4))

if cap_u:
  cap_u = np.concatenate(cap_u)
  cap_v = np.concatenate(cap_v)
  cap_jaccard = exact_jaccard(X, set_sizes, cap_u, cap_v)
  share_above_02 = float((cap_jaccard >= 0.2).mean())
  share_above_threshold = float((cap_jaccard >= lsh_threshold).mean())

  print ("\nExact Jaccard of", len(cap_jaccard), "sampled pairs from oversized buckets:")
  print (pd.Series(cap_jaccard).describe(percentiles = [0.5, 0.9, 0.99]))
  print ("Share with Jaccard >= 0.2:\t\t\t", round(share_above_02, 4))
  print ("Share with Jaccard >= LSH threshold (", round(lsh_threshold, 3), "):\t", round(share_above_threshold, 4))

  plt.figure(figsize = (8, 5))
  plt.hist(cap_jaccard, bins = 50)
  plt.axvline(lsh_threshold, color = "red", linestyle = "--", label = f"LSH threshold ~= {lsh_threshold:.2f}")
  plt.xlabel("Exact Jaccard similarity")
  plt.ylabel("Sampled pairs")
  plt.title("Pairs dropped by the bucket cap")
  plt.legend()
  plt.grid(alpha = 0.3)
  plt.show()
else:
  # No bucket exceeded the cap (e.g. on a small subset of users), so nothing was dropped.
  cap_u = np.empty(0, dtype = np.int64)
  cap_v = np.empty(0, dtype = np.int64)
  cap_jaccard = np.empty(0, dtype = np.float32)
  share_above_02 = np.nan
  share_above_threshold = np.nan
  print ("\nNo bucket exceeded the cap, so no pairs were dropped.")

bucket_cap_results = pd.DataFrame([{
    "bands": BANDS, "rows": ROWS, "max_bucket_size": MAX_BUCKET_SIZE, "lsh_threshold": lsh_threshold,
    "oversized_buckets": oversized_buckets,
    "pair_slots_dropped": int(dropped_slots), "pair_slots_kept": int(kept_slots), "dropped_share": float(dropped_share),
    "sampled_pairs": len(cap_jaccard),
    "median_jaccard": float(np.median(cap_jaccard)) if len(cap_jaccard) else np.nan,
    "p90_jaccard": float(np.percentile(cap_jaccard, 90)) if len(cap_jaccard) else np.nan,
    "p99_jaccard": float(np.percentile(cap_jaccard, 99)) if len(cap_jaccard) else np.nan,
    "share_jaccard_ge_0.2": share_above_02,
    "share_jaccard_ge_lsh_threshold": share_above_threshold}])
display (bucket_cap_results)

os.makedirs(results_path, exist_ok = True)
bucket_cap_results.to_csv(os.path.join(results_path, "bucket_cap_check.csv"), index = False)

How to read this result: the dropped share tells us how much work the cap saves, and the Jaccard distribution of the sampled pairs tells us what that work would have found.

If almost none of the sampled pairs reach the LSH threshold, the oversized buckets are made of loose overlaps caused by very popular products (like bananas), which many otherwise unrelated users buy. In that case the cap removes a large share of the candidate pairs to check while barely lowering recall, so it is a good trade-off. If a noticeable share of the sampled pairs were above the threshold, MAX_BUCKET_SIZE should be raised, or the most popular products handled separately.

## Experiment 3: Runtime and peak memory as the number of users grows

We run the whole pipeline (MinHash, LSH banding and the exact check of candidates) on 10K, 50K, 100K and all users, and record the time and peak memory of each stage. Peak memory is measured with tracemalloc, which tracks memory allocated by Python and numpy (this slows the run down slightly).

This cell runs the full pipeline several times, so it can take a few minutes.

In [ ]:
# Experiment 3: runtime and peak memory of the pipeline for a growing number of users.

USER_COUNTS = [count for count in [10_000, 50_000, 100_000] if count < n_users] + [n_users]
rng_exp3 = np.random.default_rng(SEED + 2)

def timed(function, *args):
  # Run a function and return its result, time in seconds, and peak memory in MB.
  tracemalloc.start()
  start_time = time.perf_counter()
  result = function(*args)
  elapsed = time.perf_counter() - start_time
  _, peak = tracemalloc.get_traced_memory()
  tracemalloc.stop()
  return result, elapsed, peak / 1e6

results = []
for count in USER_COUNTS:
  rows = np.sort(rng_exp3.choice(n_users, size = count, replace = False))
  items, item_offsets, sizes = subset_sets(rows)
  X_rows = X[rows]

  sig, minhash_time, minhash_memory = timed(minhash_signatures, items, item_offsets, sizes, N_HASHES)
  (u, v, _), lsh_time, lsh_memory = timed(lsh_candidate_pairs, sig, BANDS, ROWS)
  _, check_time, check_memory = timed(exact_jaccard, X_rows, sizes, u, v)

  results.append({"users": count, "candidate_pairs": len(u),
                  "minhash_s": minhash_time, "lsh_s": lsh_time, "check_s": check_time,
                  "total_s": minhash_time + lsh_time + check_time,
                  "peak_MB": max(minhash_memory, lsh_memory, check_memory)})
  print ("Finished", count, "users in", round(results[-1]["total_s"], 1), "seconds")
  del sig, u, v, X_rows

scaling_results = pd.DataFrame(results)
display (scaling_results)

os.makedirs(results_path, exist_ok = True)
scaling_results.to_csv(os.path.join(results_path, "scaling.csv"), index = False)

fig, axes = plt.subplots(1, 2, figsize = (11, 4))
axes[0].plot(scaling_results["users"], scaling_results["total_s"], marker = "o")
axes[0].set_xlabel("Users")
axes[0].set_ylabel("Total time (seconds)")
axes[0].set_title("Runtime")
axes[1].plot(scaling_results["users"], scaling_results["peak_MB"], marker = "o")
axes[1].set_xlabel("Users")
axes[1].set_ylabel("Peak memory (MB)")
axes[1].set_title("Peak memory")
for ax in axes:
  ax.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

## Link to MapReduce

This pipeline maps naturally onto MapReduce (step 5):

- **Map 1 (signatures):** each user's set is hashed independently, so computing MinHash signatures is a map step that emits (user, signature). No communication between users is needed.
- **Map 2 (banding):** for each user and each band, emit ((band, bucket hash), user).
- **Reduce (candidates):** group by (band, bucket). Every reducer receives the users in one bucket and emits all pairs among them as candidate pairs (skipping oversized buckets).
- **Final step:** remove duplicate pairs found in several bands, then compute the exact Jaccard similarity of each candidate pair and keep the top-K neighbours per user.

In [ ]:
# Clear RAM once the file is finished running

import gc

del user_products, products_lookup, user_items, flat_items, offsets, set_sizes, X
del signatures, cand_u, cand_v, cand_jaccard, similar_users
del rng_exp1, rng_exp2, rng_exp3, rng_cap
del pair_u, pair_v, true_jaccard, brute_jaccard, true_pairs
del cap_u, cap_v, cap_jaccard

gc.collect()